In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import when, col, lit, 
from pyspark.sql.types import *
import os
import glob

BRONZE_PATH = "/data/bronze/bikes/*.parquet"

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("bike-bronze-ingestion")
    .getOrCreate()
)

bikes_df = spark.read.format("parquet").load(BRONZE_PATH)

Notebook workdir: /home/jovyan/work
Files found at /data/bronze/bikes/*.parquet: 12


In [3]:
bikes_df = spark.read.format("parquet").load(BRONZE_PATH)
bikes_df.printSchema()

root
 |-- Rental Id: string (nullable = true)
 |-- Duration: string (nullable = true)
 |-- Bike Id: string (nullable = true)
 |-- End Date: string (nullable = true)
 |-- EndStation Id: string (nullable = true)
 |-- EndStation Name: string (nullable = true)
 |-- Start Date: string (nullable = true)
 |-- StartStation Id: string (nullable = true)
 |-- StartStation Name: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- load_date: date (nullable = true)
 |-- _partition_date: date (nullable = true)

+---------+--------+-------+----------------+-------------+--------------------+----------------+---------------+--------------------+--------------------+--------------------+----------+---------------+
|Rental Id|Duration|Bike Id|        End Date|EndStation Id|     EndStation Name|      Start Date|StartStation Id|   StartStation Name|         source_file| ingestion_timestamp| load_date|_partition_date|
+---------+----

In [15]:
from pyspark.sql import SparkSession

BRONZE_PATH = "/data/bronze/bikes"
SILVER_PATH = "/data/silver/bikes"
QUARANTINE_PATH = "/data/quarantine/bikes"

df = spark.read.parquet(f"{BRONZE_PATH}/*.parquet")

# Data Quality Requirements
invalid_df = (
    df.withColumn(
        "rejection_reason",
        when(col("Rental Id").isNull(), lit("missing_rental_id"))
        .when(col("Duration") <= 0, lit("invalid_duration"))
        .when(col("Start Date").isNull(), lit("missing_start_date"))
        .when(col("End Date").isNull(), lit("missing_end_date"))
        .when(col("StartStation Name").isNull(), lit("missing_start_station"))
        .when(col("EndStation Name").isNull(), lit("missing_end_station"))
    ).filter(col("rejection_reason").isNotNull())
)

valid_df = (
    df.withColumnsRenamed({
        "Rental Id": "rental_id",
        "Duration": "duration",
        "Start Date": "start_date",
        "End Date": "end_date",
        "EndStation Id": "end_station_id",
        "StartStation Id": "start_station_id",
        "StartStation Name": "start_station_name",
        "EndStation Name": "end_station_name",
        "EndStation Id": "end_station_id",
        "Bike Id": "bike_id"
    })
    .filter(col("Rental Id").isNotNull())
    .filter(col("Duration") > 0)
    .filter(col("Start Date").isNotNull())
    .filter(col("End Date").isNotNull())
    .filter(col("StartStation Name").isNotNull())
    .filter(col("EndStation Name").isNotNull())
)

valid_df = valid_df.dropDuplicates(["rental_id"])
valid_df.write.mode("overwrite").parquet(SILVER_PATH)
invalid_df.write.mode("overwrite").parquet(QUARANTINE_PATH)

print(f"Silver rows: {valid_df.count()}")
print(f"Quarantined rows: {invalid_df.count()}")

+---------+--------+-------+----------------+--------------+--------------------+----------------+----------------+--------------------+--------------------+--------------------+----------+---------------+
|rental_id|duration|bike_id|        end_date|end_station_id|    end_station_name|      start_date|start_station_id|  start_station_name|         source_file| ingestion_timestamp| load_date|_partition_date|
+---------+--------+-------+----------------+--------------+--------------------+----------------+----------------+--------------------+--------------------+--------------------+----------+---------------+
| 47984313|    1740|    487|20/09/2015 00:29|            83|Panton Street, We...|20/09/2015 00:00|               9|New Globe Walk, B...|file:///data/sour...|2026-11-16 12:45:...|2026-11-16|           NULL|
| 47984319|    1380|   9571|20/09/2015 00:24|           204|Margery Street, C...|20/09/2015 00:01|             217|Wormwood Street, ...|file:///data/sour...|2026-11-16 12:45:..

In [41]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import when, col, lit, to_timestamp

BRONZE_PATH = "/data/bronze/bikes"
SILVER_PATH = "/data/silver/bikes"
QUARANTINE_PATH = "/data/quarantine/bikes"

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("bike-bronze-ingestion")
    .getOrCreate()
)

df = spark.read.parquet(f"{BRONZE_PATH}/*.parquet")

# DATA QUALTIY REQUIREMENTS 
invalid_df = (
    df.withColumn(
        "rejection_reason",
        when(col("Rental Id").isNull(), lit("missing_rental_id"))
        .when(col("Duration") <= 0, lit("invalid_duration"))
        .when(col("Start Date").isNull(), lit("missing_start_date"))
        .when(col("End Date").isNull(), lit("missing_end_date"))
        .when(col("StartStation Name").isNull(), lit("missing_start_station"))
        .when(col("EndStation Name").isNull(), lit("missing_end_station"))
    )
    .filter(col("rejection_reason").isNotNull())
)

valid_df = (
    df.withColumnsRenamed({
        "Rental Id": "rental_id",
        "Duration": "duration",
        "Start Date": "start_date",
        "End Date": "end_date",
        "EndStation Id": "end_station_id",
        "StartStation Id": "start_station_id",
        "StartStation Name": "start_station_name",
        "EndStation Name": "end_station_name",
        "EndStation Id": "end_station_id",
        "Bike Id": "bike_id"
    })
    .filter(col("Rental Id").isNotNull())
    .filter(col("Duration") > 0)
    .filter(col("Start Date").isNotNull())
    .filter(col("End Date").isNotNull())
    .filter(col("StartStation Name").isNotNull())
    .filter(col("EndStation Name").isNotNull())
    .select(
        "rental_id",
        "bike_id",
        to_timestamp(col("start_date"), 'dd/MM/yyyy HH:mm').alias("start_date"),
        to_timestamp(col("end_date"), 'dd/MM/yyyy HH:mm').alias("end_date"),
        col("duration").cast("int").alias("duration"),
        "start_station_id",
        "start_station_name",
        "end_station_id",
        "end_station_name",
        "ingestion_timestamp",
        "load_date"
    )
)

valid_df = valid_df.dropDuplicates(["rental_id"])
valid_df.write.mode("overwrite").parquet(SILVER_PATH)
invalid_df.write.mode("overwrite").parquet(QUARANTINE_PATH)